# Pairs Trading: Exercise Workbook

**Instructions:** Fill in the markdown prompts using your notes. Fill in the missing code `___` to complete the programming exercises.

## Data Download
First, we download a sample universe of 10 stocks across different industries.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from itertools import combinations

# 10 Stock Universe grouped by Industry
industry_map = {
    'Tech': ['AAPL', 'MSFT', 'GOOGL', 'META'],
    'Financials': ['JPM', 'BAC', 'WFC'],
    'Energy': ['XOM', 'CVX', 'COP']
}
tickers = [ticker for group in industry_map.values() for ticker in group]

print("Downloading data...")
data = yf.download(tickers, start="2021-01-01", end="2024-01-01")['Close']
data = data.dropna()
print("Data downloaded.")


## Stationarity & Unit Root Tests
Before we can trade a pair, we must ensure the relationship between the two assets is stable over time.

### 📝 Explanation: The ADF Test
*Explain what the Augmented Dickey-Fuller test is and state its null hypothesis here.*

### 📝 Explanation: The KPSS Test
*Explain what the KPSS test is and how its null hypothesis differs from the ADF test here.*

### 💻 Code: Stationarity Testing

In [ ]:
# Extract AAPL prices
aapl_prices = data['AAPL']

# TODO: Run the ADF test on AAPL prices
adf_result = ___(aapl_prices)
print("ADF p-value:", adf_result[1])

# TODO: Run the KPSS test on AAPL prices
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    kpss_result = ___(aapl_prices)
print("KPSS p-value:", kpss_result[1])


## Pair Selection (Industry-Based)
Generate all unique pairs within the same industry to ensure they share fundamental risk drivers.

In [ ]:
industry_pairs = []

# TODO: Iterate through `industry_map` and use combinations to create pairs
for industry, group_tickers in industry_map.items():
    pairs = list(combinations(___, ___))
    industry_pairs.extend(pairs)

print(f"Total intra-industry pairs generated: {len(industry_pairs)}")
print(industry_pairs[:5])


## Spread Construction & Hedge Ratio

### 📝 Explanation: Log Prices
*Why do we use log prices instead of raw prices for spread construction? Explain here.*

### 💻 Code: Spread Construction (OLS)
Let's test the pair AAPL and MSFT.

In [ ]:
# TODO: Calculate log prices for AAPL and MSFT
log_aapl = np.log(___)
log_msft = np.log(___)

# TODO: Calculate Hedge Ratio using OLS (Y = AAPL, X = MSFT)
# Remember to add a constant to X using sm.add_constant()
X = sm.add_constant(___)
model = sm.OLS(___, X).fit()
hedge_ratio = model.params.iloc[1]

print(f"Hedge Ratio: {hedge_ratio}")

# TODO: Calculate the spread
spread = ___ - (hedge_ratio * ___)
spread.plot(title="AAPL - MSFT Log Spread")
plt.show()


## Mean Reversion & Half-Life

### 📝 Explanation: Half-Life
*Explain what the half-life of mean reversion is and how it is calculated using an Ornstein-Uhlenbeck process.*

In [ ]:
# TODO: Calculate lagged spread and delta spread
spread_lag = spread.___()
spread_lag = spread_lag.dropna()
delta_spread = spread - ___
delta_spread = delta_spread.dropna()

# Ensure alignment
spread_lag, delta_spread = spread_lag.align(delta_spread, join='inner')

# TODO: Run OLS to find lambda (slope)
X_hl = spread_lag
model_hl = sm.OLS(delta_spread, X_hl).fit()
lambda_val = model_hl.params.iloc[0]

# TODO: Calculate half-life
half_life = -np.log(2) / ___
print(f"Half-life: {half_life} days")


## Signal Generation with Bollinger Bands

### 📝 Explanation: Bollinger Bands
*Explain how Bollinger Bands are constructed and the logic for generating Long/Short signals for a mean-reverting spread.*

In [ ]:
# TODO: Define lookback window (e.g., 20 days)
window = ___

# TODO: Calculate rolling mean and standard deviation
rolling_mean = spread.rolling(window=window).___()
rolling_std = spread.rolling(window=window).___()

# TODO: Construct Upper and Lower Bands (2 standard deviations)
upper_band = rolling_mean + (___ * rolling_std)
lower_band = rolling_mean - (___ * rolling_std)

# TODO: Generate Signals
# 1 if spread < lower_band, -1 if spread > upper_band, 0 otherwise
signals = pd.Series(0, index=spread.index)
signals[spread < ___] = 1
signals[spread > ___] = -1

# Plot
plt.figure(figsize=(10,6))
plt.plot(spread, label='Spread')
plt.plot(rolling_mean, label='Moving Average', color='black', linestyle='--')
plt.plot(upper_band, label='Upper Band', color='red', linestyle=':')
plt.plot(lower_band, label='Lower Band', color='green', linestyle=':')
plt.legend()
plt.title("Bollinger Bands for AAPL/MSFT Spread")
plt.show()
